# Persistent Append-Only Log

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Databases, Operating Systems · **Difficulty/Frequency:** Very Common (8/10)

> **Language note.** The official answer is written in Java (`FileChannel` / `force(false)`). This notebook implements the *same design* in Python so every claim below is executable: `os.pwrite`/`os.pread` stand in for positional channel I/O, and `os.fsync` stands in for `force`. The Java reference is preserved verbatim in [`README.md`](README.md).

## Concepts

**What this problem is really testing:**
- Designing a **self-describing binary file format** — bytes on disk with no schema to help you
- Knowing what **durability** actually costs, and the difference between "written" and "on disk"
- Turning a file into a **log-structured store**: the foundation of write-ahead logs, Kafka, and LSM-tree databases like MongoDB's WiredTiger

**First-principles primer — what is each piece?**

- **Append-only log** — a file you only ever add to the end of; you never overwrite or delete in place. That single restriction buys you a lot: writes are sequential (the fastest thing a disk does), old data never moves so **byte offsets stay valid forever**, and there is no partial-update corruption to reason about.
- **Offset** — the byte position where a record starts. Because the file only grows, an offset is a permanent address. `append` returns one; `read` takes one. This is exactly what a database calls a *record pointer* or *RID*.
- **Length prefix (framing)** — a raw file is just bytes; nothing marks where one record stops. Writing `[4-byte length][payload]` makes the file **self-describing**: read 4 bytes, and they tell you how many more to read. This is *framing*, the same problem TCP protocols solve.
- **Big-endian** — the byte order for the length. Most-significant byte first, which is the network/portable convention, so a log written on one machine reads correctly on another. Python: `struct.pack(">I", n)`.
- **The buffer/page-cache trap** — when `write()` returns, your bytes are usually in the OS **page cache** (RAM), not on the platter. A process crash is survivable (the OS still has them); a *power loss* is not. **`fsync`** is the call that blocks until the data is physically durable. It is expensive — often milliseconds — which is why real systems batch it.

**The three-way cost triangle:**

| Design choice | Why |
|---|---|
| One `write` per record (length + payload in **one** buffer) | Two writes leave a window where a crash yields a length prefix with no payload — a structurally corrupt file |
| Keep `size` in a **variable**, not `os.path.getsize()` | Asking the OS for the file size is a syscall; doing it per append turns O(1) into "cheap but needless overhead" |
| Validate the length before allocating | A corrupt 4 bytes can decode to 2 GB and blow up memory; check it against the file size first |

**Simple worked example.** Append `b"hi"` then `b"abc"` to an empty log:

```
offset 0:  00 00 00 02 | 'h' 'i'          <- returns offset 0, size becomes 6
offset 6:  00 00 00 03 | 'a' 'b' 'c'      <- returns offset 6, size becomes 13
```

`read(6)`: read 4 bytes at 6 → `3`; read 3 bytes at 10 → `b"abc"`. No index needed — the file explained itself.

## Problem Statement

Implement an `AppendOnlyLog` over a single file:

| Method | Behaviour |
|---|---|
| `AppendOnlyLog(path)` | Open or create the log; be ready to append to an existing one |
| `append(record) -> offset` | Write a record, return the byte offset it starts at. **O(1) amortized** |
| `read(offset) -> bytes` | Return the record starting at that offset |
| `scan(start_offset)` | Lazily iterate every record from `start_offset` to the end |
| `flush()` | Guarantee everything appended so far survives a crash |
| `close()` | Release the file |

**Requirements**

- Each record is framed as a **4-byte big-endian length** followed by the payload.
- After a crash and restart, every record that was `flush()`ed must be recoverable.

**Follow-ups:** log compaction to reclaim space; using the log as the backbone of a key-value store with an in-memory hash index of offsets.

### Approach 1 — Naive (no framing discipline, no size tracking)

**Idea:** the version most people write first. It "works" on the happy path but breaks all three requirements:

1. **Two separate writes** for the length and the payload — a crash between them leaves a length prefix pointing at bytes that do not exist.
2. **`os.path.getsize()` on every append** to find the end — a syscall per record instead of an in-memory counter.
3. **`read` addressed by record *number*, not byte offset** — so finding record `k` means walking the file from the start.

**Time complexity:** O(1) per `append` but with a syscall's constant; **O(k) per `read`** — every lookup rescans the prefix of the file.

**Space complexity:** O(1) in memory.

In [ ]:
import os
import struct
from typing import Iterator, List, Optional

HEADER = struct.Struct(">I")          # 4-byte big-endian unsigned length


class NaiveLog:
    """Baseline: correct-ish, but O(k) reads and a torn-write window on append."""

    def __init__(self, path: str) -> None:
        self.path = path
        if not os.path.exists(path):
            open(path, "wb").close()
        self.f = open(path, "r+b")

    def append(self, record: bytes) -> int:
        offset = os.path.getsize(self.path)   # a syscall on EVERY append
        self.f.seek(0, os.SEEK_END)
        self.f.write(HEADER.pack(len(record)))
        self.f.write(record)                  # TORN-WRITE WINDOW: a crash here leaves a header alone
        self.f.flush()
        return offset

    def read_nth(self, k: int) -> bytes:
        """Read the k-th record by walking the file from the beginning: O(k)."""
        self.f.seek(0)
        for _ in range(k):
            head = self.f.read(4)
            if len(head) < 4:
                raise IndexError("record index out of range")
            self.f.seek(HEADER.unpack(head)[0], os.SEEK_CUR)
        head = self.f.read(4)
        if len(head) < 4:
            raise IndexError("record index out of range")
        return self.f.read(HEADER.unpack(head)[0])

    def close(self) -> None:
        self.f.close()

### Approach 2 — Optimal (single framed write, in-memory size, positional reads, real `fsync`)

**Idea:** four changes turn the baseline into a real log.

1. **One write per record.** Build `[length][payload]` in a single `bytes` object and hand it to one `os.pwrite`. There is no longer any moment where a header exists without its payload.
2. **Track `size` in memory.** Seeded from the real file size in the constructor (so reopening an existing log works), then incremented by exactly `4 + len(record)`. Correct forever, because an append-only file never shrinks.
3. **Positional reads.** Reading at an arbitrary offset never moves a shared file pointer, so `read` and `scan` cannot corrupt each other's position — and `read` is O(1) regardless of how far into the file the offset is. On POSIX this is `os.pread`/`os.pwrite`, the direct analogue of Java's `FileChannel.read(buf, position)`; Windows has no such call, so `_pread`/`_pwrite` below fall back to `os.lseek` + `os.read`/`os.write`. **The fallback is not equivalent under concurrency** — it moves the shared file pointer, so concurrent callers would interleave seeks. That is exactly why the real answer to "how do you handle concurrent readers and writers?" is either genuine positional I/O or one file descriptor per reader.
4. **`flush()` = `os.fsync`.** `f.flush()` only pushes Python's buffer into the OS; `os.fsync` is what forces the OS's page cache onto the disk. This is the Python counterpart of Java's `force(false)` — data blocks only, no metadata sync.

**Validation matters:** before allocating a payload buffer, check that the decoded length is sane against the known file size. A corrupt header could otherwise ask for gigabytes.

**Time complexity:** **O(1) amortized** per `append`; **O(1)** per `read`; O(n) to `scan` n records.

**Space complexity:** O(len(record)) transient per call; **O(1)** persistent — the log lives on disk, never in memory.

In [ ]:
HAS_POSITIONAL = hasattr(os, "pwrite")     # POSIX has pread/pwrite; Windows does not


def _pwrite(fd: int, data: bytes, offset: int) -> int:
    """Write at an absolute offset. Positional where available, seek+write otherwise."""
    if HAS_POSITIONAL:
        return os.pwrite(fd, data, offset)
    os.lseek(fd, offset, os.SEEK_SET)      # NOT concurrency-safe: moves the shared pointer
    return os.write(fd, data)


def _pread(fd: int, n: int, offset: int) -> bytes:
    """Read n bytes at an absolute offset."""
    if HAS_POSITIONAL:
        return os.pread(fd, n, offset)
    os.lseek(fd, offset, os.SEEK_SET)
    return os.read(fd, n)


class AppendOnlyLog:
    """Length-prefixed append-only log with O(1) reads and explicit durability."""

    def __init__(self, path: str) -> None:
        self.path = path
        # os.open with O_CREAT gives us a raw fd for pread/pwrite (positional, pointer-free).
        flags = os.O_RDWR | os.O_CREAT
        if hasattr(os, "O_BINARY"):           # Windows: don't translate newlines in binary data
            flags |= os.O_BINARY
        self.fd = os.open(path, flags, 0o644)
        self.size = os.fstat(self.fd).st_size  # reopening an existing log just works
        self.closed = False

    # ---- writes ----------------------------------------------------------
    def append(self, record: bytes) -> int:
        self._ensure_open()
        offset = self.size
        blob = HEADER.pack(len(record)) + record   # ONE buffer => no torn header/payload
        written = 0
        while written < len(blob):                 # pwrite may write short; loop until done
            written += _pwrite(self.fd, blob[written:], offset + written)
        self.size += len(blob)                     # in-memory counter, no getsize() syscall
        return offset

    def flush(self) -> None:
        """Block until every appended byte is on stable storage (Java's force(false))."""
        self._ensure_open()
        os.fsync(self.fd)

    # ---- reads -----------------------------------------------------------
    def _pread_exact(self, offset: int, n: int) -> bytes:
        buf = b""
        while len(buf) < n:
            chunk = _pread(self.fd, n - len(buf), offset + len(buf))
            if not chunk:
                raise IOError(f"unexpected EOF at {offset + len(buf)}")
            buf += chunk
        return buf

    def read(self, offset: int) -> bytes:
        self._ensure_open()
        if offset < 0 or offset >= self.size:
            raise ValueError(f"Offset out of range: {offset}")
        length = HEADER.unpack(self._pread_exact(offset, 4))[0]
        # Validate BEFORE allocating: a corrupt header could ask for gigabytes.
        if offset + 4 + length > self.size:
            raise IOError(f"Corrupt record at offset {offset}")
        return self._pread_exact(offset + 4, length)

    def scan(self, start_offset: int = 0) -> Iterator[bytes]:
        """Lazy generator - a multi-gigabyte log costs one record of memory."""
        self._ensure_open()
        cursor = start_offset
        while cursor < self.size:
            rec = self.read(cursor)
            cursor += 4 + len(rec)             # every record occupies exactly 4 + len bytes
            yield rec

    # ---- lifecycle -------------------------------------------------------
    def close(self) -> None:
        if not self.closed:
            self.closed = True
            os.close(self.fd)

    def _ensure_open(self) -> None:
        if self.closed:
            raise ValueError("Log is closed")

### Follow-up 1 — a key-value store on top of the log (hash index of offsets)

**Idea:** the log gives you durability and ordering; a hash map gives you lookup. Combine them:

- `put(k, v)` → append a `(key, value)` record, then set `index[k] = offset`. Older versions of the key stay in the file but become **unreachable garbage**.
- `get(k)` → one dict lookup for the offset, one `read` — O(1).
- `delete(k)` → append a **tombstone** (a record that means "this key is gone") and drop the key from the index. You cannot erase in place in an append-only file, so deletion must itself be an append.
- **Recovery** → on startup, `scan()` the whole log from the beginning and replay it into the index. Because records appear in write order, the *last* record for a key wins — which is exactly the state the store was in.

This is [Bitcask](https://riak.com/assets/bitcask-intro.pdf) in miniature, and the same shape as a database's write-ahead log plus its in-memory index.

**Time complexity:** O(1) for `put`, `get`, `delete`; O(n) once for recovery.

**Space complexity:** O(number of live keys) in memory — values stay on disk.

In [ ]:
import json


class LogKV:
    """Bitcask-style KV store: append-only log for data, in-memory dict for offsets."""

    TOMBSTONE = b"\x00__deleted__"

    def __init__(self, path: str) -> None:
        self.log = AppendOnlyLog(path)
        self.index = {}                       # key -> byte offset of its newest record
        self._recover()

    @staticmethod
    def _encode(key: str, value: bytes) -> bytes:
        kb = key.encode()
        return HEADER.pack(len(kb)) + kb + value      # framed key, then the raw value

    @staticmethod
    def _decode(rec: bytes):
        klen = HEADER.unpack(rec[:4])[0]
        return rec[4:4 + klen].decode(), rec[4 + klen:]

    def _recover(self) -> None:
        """Replay the log in write order; the LAST record for a key wins."""
        offset = 0
        for rec in self.log.scan(0):
            key, value = self._decode(rec)
            if value == self.TOMBSTONE:
                self.index.pop(key, None)     # a delete that was recorded before the crash
            else:
                self.index[key] = offset
            offset += 4 + len(rec)

    def put(self, key: str, value: bytes) -> None:
        self.index[key] = self.log.append(self._encode(key, value))

    def get(self, key: str) -> Optional[bytes]:
        if key not in self.index:
            return None
        _, value = self._decode(self.log.read(self.index[key]))
        return value

    def delete(self, key: str) -> None:
        if key in self.index:
            self.log.append(self._encode(key, self.TOMBSTONE))   # deletion is an append too
            del self.index[key]

    def flush(self) -> None:
        self.log.flush()

    def close(self) -> None:
        self.log.close()

### Follow-up 2 — compaction (reclaim the space dead records occupy)

**Idea:** every overwrite and delete leaves the old record stranded in the file. Compaction rewrites **only the live records** into a fresh file and swaps it in.

The two things that make it safe:

1. **Write to a temp file, then `os.replace`.** On both POSIX and Windows, `os.replace` is atomic — at every instant, readers see either the complete old file or the complete new one, never a half-written one. Compaction is therefore crash-safe: crash mid-rewrite and you still have the original.
2. **Offsets change, so the index must be rebuilt.** This is the answer to the follow-up "what happens to the hash index after compaction?" — you record each record's *new* offset as you write it and swap the whole index in at the end. (The alternative, an old→new translation table, only postpones the same work.)

Live records are identified by the index: an offset present in `index.values()` is live; everything else is garbage.

**Time complexity:** O(live bytes) to rewrite.

**Space complexity:** O(live bytes) transient on disk (both copies exist briefly).

In [ ]:
def compact(kv: LogKV) -> int:
    """Rewrite only the live records into a fresh file; return bytes reclaimed."""
    old_size = kv.log.size
    tmp_path = kv.log.path + ".compact"
    if os.path.exists(tmp_path):
        os.remove(tmp_path)

    new_log = AppendOnlyLog(tmp_path)
    new_index = {}
    for key, offset in kv.index.items():          # only offsets still in the index are live
        rec = kv.log.read(offset)
        new_index[key] = new_log.append(rec)      # note the NEW offset as we go
    new_log.flush()
    new_log.close()
    kv.log.close()

    os.replace(tmp_path, kv.log.path)             # ATOMIC swap - crash-safe at every instant
    kv.log = AppendOnlyLog(kv.log.path)
    kv.index = new_index                          # the whole index is replaced, not patched
    return old_size - kv.log.size

## Verification

Exercise the framing, offset arithmetic, laziness, durability, recovery, and compaction — all against real files in a temporary directory.

In [ ]:
import tempfile
import shutil

tmpdir = tempfile.mkdtemp(prefix="aol_")
p = os.path.join(tmpdir, "test.log")

# --- Framing and offsets ---
log = AppendOnlyLog(p)
o1 = log.append(b"hi")
o2 = log.append(b"abc")
o3 = log.append(b"")                       # a ZERO-LENGTH record is valid, not a sentinel
assert (o1, o2, o3) == (0, 6, 13), (o1, o2, o3)
assert log.size == 17
assert log.read(o1) == b"hi"
assert log.read(o2) == b"abc"
assert log.read(o3) == b""
assert list(log.scan(0)) == [b"hi", b"abc", b""]
assert list(log.scan(o2)) == [b"abc", b""]   # scanning from a mid-file offset

# Out-of-range offsets are rejected rather than returning garbage
for bad in (-1, log.size, log.size + 100):
    try:
        log.read(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"read({bad}) should have raised")

# scan is lazy: taking one record must not materialise the rest
gen = log.scan(0)
assert next(gen) == b"hi"
gen.close()

# --- Durability + reopening an existing log ---
log.flush()
log.close()

reopened = AppendOnlyLog(p)                 # size is recovered from the file itself
assert reopened.size == 17
assert list(reopened.scan(0)) == [b"hi", b"abc", b""]
o4 = reopened.append(b"after reopen")       # appending to an existing log continues cleanly
assert reopened.read(o4) == b"after reopen"
reopened.flush()
reopened.close()

# Simulate a crash: a NEW process opening the file sees every flushed record
survivor = AppendOnlyLog(p)
assert list(survivor.scan(0)) == [b"hi", b"abc", b"", b"after reopen"]
survivor.close()

# Closed logs reject further use
survivor2 = AppendOnlyLog(p)
survivor2.close()
try:
    survivor2.append(b"x")
except ValueError:
    pass
else:
    raise AssertionError("append on a closed log should raise")

# --- Naive and optimal implementations agree on content ---
p2 = os.path.join(tmpdir, "cmp.log")
p3 = os.path.join(tmpdir, "cmp2.log")
records = [f"record-{i}".encode() for i in range(100)] + [b"", b"x" * 500]
naive, fast = NaiveLog(p2), AppendOnlyLog(p3)
for r in records:
    naive.append(r)
    fast.append(r)
assert [naive.read_nth(i) for i in range(len(records))] == records
assert list(fast.scan(0)) == records
naive.close()
fast.close()

# --- Key-value store on top of the log ---
kvp = os.path.join(tmpdir, "kv.log")
kv = LogKV(kvp)
kv.put("a", b"1")
kv.put("b", b"2")
kv.put("a", b"3")                            # overwrite: old record becomes garbage
assert kv.get("a") == b"3"
assert kv.get("b") == b"2"
assert kv.get("missing") is None
kv.delete("b")
assert kv.get("b") is None
kv.flush()
kv.close()

# Recovery: a fresh process replays the log and lands on the same state
kv2 = LogKV(kvp)
assert kv2.get("a") == b"3", "newest record for a key must win on replay"
assert kv2.get("b") is None, "a tombstone must survive recovery"

# --- Compaction ---
for i in range(200):
    kv2.put("churn", str(i).encode())        # 200 versions, 199 of them garbage
size_before = kv2.log.size
reclaimed = compact(kv2)
assert reclaimed > 0, "compaction must reclaim space"
assert kv2.log.size < size_before
assert kv2.get("a") == b"3"                  # data survives...
assert kv2.get("churn") == b"199"            # ...at its newest version
assert kv2.get("b") is None                  # ...and deletions stay deleted
kv2.close()

# Compaction survives a restart: the rebuilt index matches the compacted file
kv3 = LogKV(kvp)
assert kv3.get("a") == b"3"
assert kv3.get("churn") == b"199"
kv3.close()

shutil.rmtree(tmpdir, ignore_errors=True)
print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Concurrent readers and writers.** True positional I/O (`os.pread`/`os.pwrite`, Java's `FileChannel.read(buf, pos)`) does not share a file pointer, so **readers need no lock at all** — a reader at a stale offset simply reads an older record. The writer is the problem: `offset = self.size` followed by `self.size += n` is a read-modify-write race, so two appends could claim the same offset. Guard just those two lines with a mutex (the actual `pwrite` can stay outside it, since each writer owns a disjoint byte range), or funnel all appends through a single writer thread — the usual choice, because it also lets you batch `fsync` calls.
- **Batching `fsync`.** `fsync` costs milliseconds; calling it per append caps you at a few hundred writes/second. Real systems offer a *group commit*: many appends, one `fsync`, and every waiting writer is acknowledged together. That is a durability/latency dial, and being able to name the trade is the point.
- **Records larger than 4 GB.** `>I` caps a record at 2^32−1 bytes. Options: an 8-byte length (costs 4 extra bytes on *every* record), or a **varint** length (1 byte for small records, more for large) — cheaper on average but the header is no longer fixed-size, so you can no longer read "the next 4 bytes" blindly, and backwards scanning gets harder. Format changes like this need a version byte in a file header.
- **Detecting corruption, not just bad lengths.** The length check catches structural damage but not a flipped bit inside a payload. Add a **CRC32** per record (`[len][crc][payload]`) and verify on read. This is what Kafka and WiredTiger do; it is the difference between "the file parses" and "the data is right".
- **Compaction while the log is live.** The version here stops the world. Production systems instead seal the current file and start a **new active segment**, then compact the sealed segments in the background — writes never block, and only the segment being replaced needs an index swap.

## Empirical complexity check

Compare **reading by record index with a scan from the start** (Approach 1, O(k) per read) against **reading by byte offset** (Approach 2, O(1) per read). A fixed number of random records is read from a log whose length doubles.

| Growth when the record count doubles | What it means |
|---|---|
| ~2x | linear — each read walks an ever-longer prefix of the file |
| ~1x | constant — a positional read costs the same at byte 40 or byte 40,000,000 |

This is the concrete payoff of `append` returning an **offset** instead of an index: the offset is a direct address, so no walking is ever required.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random
import tempfile
import shutil

BENCH_DIR = tempfile.mkdtemp(prefix="aol_bench_")
READS = 200


_run = [0]


def make_logs(n):
    """Build one naive log and one offset log holding n identical records."""
    _run[0] += 1                                   # unique names: never reopen a file still held open
    tag = f"{n}_{_run[0]}"
    pa = os.path.join(BENCH_DIR, f"naive_{tag}.log")
    pb = os.path.join(BENCH_DIR, f"fast_{tag}.log")
    naive, fast = NaiveLog(pa), AppendOnlyLog(pb)
    offsets = []
    for i in range(n):
        rec = f"payload-{i:08d}".encode()
        naive.append(rec)
        offsets.append(fast.append(rec))
    rng = random.Random(3)
    picks = [rng.randrange(n) for _ in range(READS)]
    return (naive, fast, offsets, picks)


def run_scan_reads(naive, fast, offsets, picks):
    for k in picks:
        naive.read_nth(k)          # O(k): walks the file from byte 0 every time


def run_offset_reads(naive, fast, offsets, picks):
    for k in picks:
        fast.read(offsets[k])      # O(1): jump straight to the byte


benchmark(
    {"Approach 1 - read by index (scan from start)": run_scan_reads,
     "Approach 2 - read by byte offset": run_offset_reads},
    make_logs,
    sizes=[500, 1000, 2000, 4000],
    repeats=1,
)

shutil.rmtree(BENCH_DIR, ignore_errors=True)   # best-effort: open handles may keep files around

## Patterns learned

- **Append-only turns offsets into permanent addresses.** The moment you promise never to overwrite, a byte position becomes a stable pointer — which is what makes an O(1) `read` and a tiny in-memory index possible. Most of a log-structured design's power comes from this one restriction.
- **Frame your records or you cannot parse them.** A length prefix makes a byte stream self-describing. The same problem, and the same fix, appears in network protocols, serialization formats, and every custom file format you will ever write.
- **One write, one record.** Any structure that must be all-or-nothing should reach the OS in a single call. Splitting it creates a crash window, and crash windows are the bugs you find in production at 3am.
- **"Written" ≠ "durable".** `write` fills a buffer; only `fsync` (Java: `force`) reaches the disk. Know which one your API called, and know that durability is a latency cost you can batch but not avoid.
- **Validate a length before you trust it.** Any size read *from* data is attacker- and corruption-controlled. Bound-check it against something you already know before allocating.
- **Deletion in an immutable store is an append.** Tombstones plus periodic compaction is the universal answer — it is how Cassandra, Kafka log-compaction, and MVCC databases all handle removal.
- **Atomic rename is the crash-safe way to replace a file.** Write to a temp path, `os.replace`. Readers only ever see a complete file, and a crash leaves the original intact.